In [1]:
import torch

x1 = torch.Tensor([2.0]).double() ; x1.requires_grad = True # we do double because by defaul torch stores it as a 32 float, while python's floats are 64 floats so this normalizes stuff.
x2 = torch.Tensor([0.0]).double() ; x2.requires_grad = True # we put requires grad = True because it's false by default for single value tensors (since they are leaf nodes)
w1 = torch.Tensor([-3.0]).double() ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double() ; w2.requires_grad = True

b = torch.Tensor([6.8813735870195]).double() ; b.requires_grad = True
n = x1 * w1 + x2 * w2 +b
o = torch.tanh(n)
print(o.data.item())



0.7071066904050358


In [26]:
o.backward()

print("x2", x2.grad.item())
print("w2", w2.grad.item())
print("x1", x1.grad.item())
print("w1", w1.grad.item())

x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [37]:
x1.grad

tensor([-1.5000], dtype=torch.float64)

### We create a neurone:


In [158]:
import math
import random
class Neuron:
  def __init__(self, nin):
    self.w =  [random.uniform(0,1) for _ in range(nin)]
    self.b = random.uniform(0,1)

  def __call__(self,x):
    act = sum(([wi * xi for wi,xi in zip(self.w,x)]),self.b)

    out = math.tanh(act)
    return out



  def parameters(self):
    return self.w + [self.b]





In [159]:
n = Neuron(3)
n([0.1,0.2,0.3])


0.44091431005964765

### We create a layer:

In [160]:
class Layer:
    def __init__(self, nin, nout):
      self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self,x):
      val = [n(x) for n in self.neurons]
      return val[0] if len(val) == 1 else val

    def parameters(self):
      return [p for neuron in self.neurons for p in neuron.parameters()]






In [161]:
lay = Layer(2,3)
lay([2.0,3.0])

[0.9988054487692245, 0.6794256484191716, 0.998367989929692]

### Now we create a multi layer perceptron:


In [167]:
class MLP:
  def __init__(self, nin, nouts):
    dims = [nin] + nouts
    self.layers = [Layer(dims[i],dims[i+1]) for i in range(len(nouts))]



  def __call__(self,x):
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()] #it's like we are just putting les boucles sans enters / tabs


In [168]:
x = [2.0,-3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

0.44024114441053797

In [169]:
xs= [
    [2.0,3.0,-1.0],
    [3.0,-1.0,0.5],
    [0.5,1.0,1.0],
    [1.0,1.0,-1.0]
]

ys = [1.0,-1.0,-1.0,1.0]

ypred = [n(x) for x in xs]
print(ypred)

[0.9835820625628195, 0.9710538869703991, 0.9840338110854594, 0.9787627244958108]


In [170]:
loss = sum([(ygt - yout)**2 for ygt,yout in zip(ys,ypred)]) #mse loss
print(loss)

7.822164159411943


In [172]:
n.parameters()
len(n.parameters())

41

# Making makemore

## What is makemore:
it's a character-level language model, basically you're predicting the next letter in a sequence of letters (i.e. the word).